# EDA платформы FishGrow

**Студент**: Шефер Анна Александровна, 22206  
**Исходный коммит**: 1f65547b63c3043e2dcfff0c6435afa993331d4e  
**Ветка**: lab/01-eda  
**Дата**: 2026-09-23

In [34]:
import sys
print(sys.executable)

import pandas as pd
print(f"pandas {pd.__version__}")

c:\Users\User\.virtualenvs\01-eda-HzBk4NwM\Scripts\python.EXE
pandas 3.0.6


In [35]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("../assets/data/Fish.csv")

df = pd.read_csv(DATA_PATH)
print(f"Размер таблицы: {df.shape}")
print(f"Столбцы: {list(df.columns)}")
df.head()

Размер таблицы: (159, 7)
Столбцы: ['Species', 'Weight', 'Length1', 'Length2', 'Length3', 'Height', 'Width']


,Species,Weight,Length1,Length2,Length3,Height,Width
0,Bream,242.0,23.2,25.4,30.0,11.5200,4.0200
1,Bream,290.0,24.0,26.3,31.2,12.4800,4.3056
2,Bream,340.0,23.9,26.5,31.1,12.3778,4.6961
3,Bream,363.0,26.3,29.0,33.5,12.7300,4.4555
4,Bream,430.0,26.5,29.0,34.0,12.4440,5.1340


## 1. Первичный осмотр данных

In [36]:
df.dtypes

Species        str
Weight     float64
Length1    float64
Length2    float64
Length3    float64
Height     float64
Width      float64
dtype: object

In [37]:
df['Species'].value_counts()

Species
Perch        56
Bream        35
Roach        20
Pike         17
Smelt        14
Parkki       11
Whitefish     6
Name: count, dtype: int64

In [38]:
df.describe()

,Weight,Length1,Length2,Length3,Height,Width
count,159.000000,159.000000,159.000000,159.000000,159.000000,159.000000
mean,398.326415,26.247170,28.415723,31.227044,8.970994,4.417486
std,357.978317,9.996441,10.716328,11.610246,4.286208,1.685804
min,0.000000,7.500000,8.400000,8.800000,1.728400,1.047600
25%,120.000000,19.050000,21.000000,23.150000,5.944800,3.385650
50%,273.000000,25.200000,27.300000,29.400000,7.786000,4.248500
75%,650.000000,32.700000,35.500000,39.650000,12.365900,5.584500
max,1650.000000,59.000000,63.400000,68.000000,18.957000,8.142000


In [39]:
numeric_cols = df.select_dtypes(include='number').columns
for col in numeric_cols:
    n_nonpos = (df[col] <= 0).sum()
    print(f"{col:10s} min={df[col].min():8.2f}  max={df[col].max():8.2f}  <=0: {n_nonpos}")

Weight     min=    0.00  max= 1650.00  <=0: 1
Length1    min=    7.50  max=   59.00  <=0: 0
Length2    min=    8.40  max=   63.40  <=0: 0
Length3    min=    8.80  max=   68.00  <=0: 0
Height     min=    1.73  max=   18.96  <=0: 0
Width      min=    1.05  max=    8.14  <=0: 0


In [40]:
print("Length1 < Length2:", (df['Length1'] < df['Length2']).all())
print("Length2 < Length3:", (df['Length2'] < df['Length3']).all())

Length1 < Length2: True
Length2 < Length3: True


In [41]:
row = df.iloc[0]
print(f"Length3: {row['Length3']}")
print(f"Height: {row['Height']} (если %, то {row['Height'] * row['Length3'] / 100:.2f} см)")
print(f"Width:  {row['Width']} (если %, то {row['Width'] * row['Length3'] / 100:.2f} см)")

Length3: 30.0
Height: 11.52 (если %, то 3.46 см)
Width:  4.02 (если %, то 1.21 см)


In [42]:
pike = df[df['Species'] == 'Pike'].iloc[0]
print(f"Pike: Length3={pike['Length3']}, Height={pike['Height']}, Width={pike['Width']}")
print(f"  Height / Length3 = {pike['Height'] / pike['Length3'] * 100:.1f}%")
print(f"  Width  / Length3 = {pike['Width'] / pike['Length3'] * 100:.1f}%")

Pike: Length3=34.8, Height=5.568, Width=3.3756
  Height / Length3 = 16.0%
  Width  / Length3 = 9.7%


## 2. Автоматическая проверка качества

In [43]:
import sys
sys.path.append("../src")
from data_checks import (
    check_shape_and_types,
    check_duplicates,
    check_missing_and_inf,
    check_geometric_consistency,
    check_rare_categories,
    check_near_duplicates,
    check_unique_fields,
)
print("Импорт успешен")

Импорт успешен


In [44]:
check_shape_and_types(df)

{'rows': 159,
 'cols': 7,
 'columns': ['Species',
  'Weight',
  'Length1',
  'Length2',
  'Length3',
  'Height',
  'Width'],
 'dtypes': {'Species': 'str',
  'Weight': 'float64',
  'Length1': 'float64',
  'Length2': 'float64',
  'Length3': 'float64',
  'Height': 'float64',
  'Width': 'float64'}}

In [45]:
check_duplicates(df)

{'duplicate_rows': 0, 'unique_index': True, 'index_duplicates': 0}

In [46]:
check_missing_and_inf(df)

,missing,missing_pct,n_inf,n_nonpositive,min,max
Species,0,0.0,0,0,NaN,NaN
Weight,0,0.0,0,1,0.0000,1650.000
Length1,0,0.0,0,0,7.5000,59.000
Length2,0,0.0,0,0,8.4000,63.400
Length3,0,0.0,0,0,8.8000,68.000
Height,0,0.0,0,0,1.7284,18.957
Width,0,0.0,0,0,1.0476,8.142


In [47]:
check_geometric_consistency(df)

{'Length1_lt_Length2': True,
 'Length2_lt_Length3': True,
 'all_positive': True,
 'n_violations': 0}

In [48]:
check_rare_categories(df, "Species", min_count=5)

{'n_categories': 7,
 'counts': {'Perch': 56,
  'Bream': 35,
  'Roach': 20,
  'Pike': 17,
  'Smelt': 14,
  'Parkki': 11,
  'Whitefish': 6},
 'rare': {},
 'imbalance_ratio': np.float64(9.33)}

In [49]:
check_near_duplicates(df)

{'n_near_duplicates': 0}

In [50]:
check_unique_fields(df)

{'Species': {'n_unique': 7, 'unique_ratio': 0.044, 'is_unique': False},
 'Weight': {'n_unique': 101, 'unique_ratio': 0.635, 'is_unique': False},
 'Length1': {'n_unique': 116, 'unique_ratio': 0.73, 'is_unique': False},
 'Length2': {'n_unique': 93, 'unique_ratio': 0.585, 'is_unique': False},
 'Length3': {'n_unique': 124, 'unique_ratio': 0.78, 'is_unique': False},
 'Height': {'n_unique': 154, 'unique_ratio': 0.969, 'is_unique': False},
 'Width': {'n_unique': 152, 'unique_ratio': 0.956, 'is_unique': False}}

### Выводы по проверкам

| Проверка | Результат |
|----------|-----------|
| Форма | 159 строк, 7 столбцов |
| Типы | Species — str, остальные — float64 |
| Дубликаты | 0 |
| Пропуски | 0 формальных, но 1 значение Weight = 0 |
| Бесконечности | 0 |
| Согласованность длин | Length1 < Length2 < Length3 для всех |
| Редкие категории | Нет (< 5), но дисбаланс 9.33 |
| Почти дубликаты | 0 |
| Уникальные поля | Идентификаторов нет |

**Единственная проблема**: Weight = 0 (1 строка). Обработаем в шаге 3.5.